# Chapter 14 — Problem Set 2: Solutions

Reference solutions for `problem_set_2.ipynb`.

---
*Generated by Berta AI*

In [ ]:
import os, sys, math, json
from pathlib import Path
sys.path.insert(0, os.path.join(os.getcwd(), '..', '..', 'scripts'))

import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

from peft_utils import LinearLayer, LoRALayer, apply_lora_to, merge_lora
from training_utils import EvalHarness, win_rate_stub
from dataset_utils import load_jsonl
np.random.seed(0)

## Solution 1 — LoRA Forward From Scratch

In [ ]:
def lora_forward(x, W, A, B, alpha, r):
    base = x @ W.T
    delta = (x @ A.T) @ B.T * (alpha / r)
    return base + delta

rng = np.random.default_rng(0)
base = LinearLayer.random(in_features=12, out_features=6, seed=1)
lora = apply_lora_to(base, rank=4, alpha=8.0)
lora.B = rng.normal(scale=0.1, size=lora.B.shape)
x = rng.normal(size=(3, 12))

ours = lora_forward(x, base.W, lora.A, lora.B, lora.alpha, lora.rank)
ref = lora.forward(x, base)
print('match:', np.allclose(ours, ref))

## Solution 2 — Parameter-Efficiency Ratio

In [ ]:
h = 4096
n_layers = 32
n_targets = 2  # q_proj, v_proj
total_params = 7_000_000_000
ranks = [4, 8, 16, 32]
rows = []
for r in ranks:
    per_layer = r * (h + h)
    lora_total = per_layer * n_layers * n_targets
    rows.append((r, lora_total, lora_total / total_params * 100))
for r, p, pct in rows:
    print(f'r={r:3d}  lora_params={p:>10,d}  fraction={pct:.4f}%')

plt.bar([str(r) for r, *_ in rows], [pct for *_, pct in rows])
plt.ylabel('% of 7B'); plt.xlabel('rank'); plt.title('LoRA parameter share'); plt.show()

## Solution 3 — Merge Adapters

In [ ]:
rng = np.random.default_rng(2)
base = LinearLayer.random(in_features=10, out_features=4, seed=1)
lora = apply_lora_to(base, rank=2, alpha=4.0)
lora.B = rng.normal(scale=0.1, size=lora.B.shape)
x = rng.normal(size=(5, 10))
merged = merge_lora(base, lora)
print('lora.forward == merged.forward?', np.allclose(lora.forward(x, base), merged.forward(x)))

# Two adapters: order independence
a1 = apply_lora_to(base, rank=2, alpha=4.0, seed=10); a1.B = rng.normal(scale=0.1, size=a1.B.shape)
a2 = apply_lora_to(base, rank=2, alpha=4.0, seed=20); a2.B = rng.normal(scale=0.1, size=a2.B.shape)
m12 = merge_lora(merge_lora(base, a1), a2)
m21 = merge_lora(merge_lora(base, a2), a1)
print('order independent?', np.allclose(m12.W, m21.W))
print('reason: each merge adds B@A * alpha/r to W; addition commutes.')

## Solution 4 — DPO Loss

In [ ]:
def sigmoid(z): return 1.0 / (1.0 + np.exp(-z))
def dpo_loss(logp_w, logp_l, ref_w, ref_l, beta=0.1):
    margin = beta * ((logp_w - ref_w) - (logp_l - ref_l))
    return float(-np.log(sigmoid(margin) + 1e-12).mean())

ref_w = np.array([-2.0, -3.0, -4.0])
ref_l = np.array([-3.0, -4.0, -5.0])
loss_eq = dpo_loss(ref_w, ref_l, ref_w, ref_l, beta=0.1)
print(f'policy == reference -> loss={loss_eq:.4f}  (should be ~0.6931 = -log(0.5))')

loss_better = dpo_loss(ref_w + 0.5, ref_l - 0.5, ref_w, ref_l, beta=0.1)
loss_worse = dpo_loss(ref_w - 0.5, ref_l + 0.5, ref_w, ref_l, beta=0.1)
print(f'policy moves toward chosen -> loss={loss_better:.4f}')
print(f'policy moves away from chosen -> loss={loss_worse:.4f}')
assert loss_better < loss_eq < loss_worse
print('Bounded below by 0:', loss_better >= 0)

## Solution 5 — Held-out Win Rate and Position Bias

In [ ]:
rows = load_jsonl(Path('..') / '..' / 'datasets' / 'eval_set.jsonl')
refs = [r['reference'] for r in rows]
preds_a = refs                                            # perfect
preds_b = [(r.split() + [''])[0] for r in refs]            # first token only
h = EvalHarness(references=refs)
print('A:', h.score(preds_a))
print('B:', h.score(preds_b))
ab = win_rate_stub(preds_a, preds_b, refs)
ba = win_rate_stub(preds_b, preds_a, refs)
print('A vs B:', ab)
print('B vs A:', ba)
print('Mirror symmetry:', math.isclose(ab['a_win_rate'], ba['b_win_rate']))

## Solution 6 — Registry Entry

In [ ]:
import yaml, hashlib, datetime as dt
entry = {
    'name': 'legal-summary',
    'version': '0.3.0',
    'base_model': 'meta-llama/Llama-3.2-3B',
    'adapter_path': 'adapters/legal_summary_v0_3_0.safetensors',
    'method': 'qlora',
    'dataset': {
        'name': 'legal-summary-v1',
        'size': 4200,
        'hash': hashlib.sha256(b'legal-summary-v1').hexdigest()[:12],
    },
    'hyperparams': {
        'lora_rank': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
        'lr': 2e-4, 'epochs': 3, 'batch_size': 8,
        'quantization': 'nf4', 'double_quantization': True,
    },
    'eval': {
        'held_out_f1': 0.79,
        'held_out_em': 0.42,
        'win_rate_vs_base': 0.64,
        'mmlu_delta': -0.2,
        'truthfulqa_delta': +0.1,
    },
    'gate': 'promote to prod when: F1 >= 0.78, win_rate >= 0.55, mmlu_delta >= -1.0',
    'status': 'staging',
    'owner': 'practitioner@berta.ai',
    'created_at': dt.date(2026, 5, 9).isoformat(),
}
print(yaml.safe_dump(entry, sort_keys=False))

---
*Generated by Berta AI*